# Earnings Call Lie Detector — Exploratory Data Analysis

This notebook explores the audio features and ground truth labels to validate
the core premise: **do vocal characteristics differ between executives who
meet their guidance and those who miss it?**

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pointbiserialr
import warnings

warnings.filterwarnings('ignore')

# Plotting defaults
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

LABEL_NAMES = {0: 'Met Guidance', 1: 'Missed Guidance'}
COLORS = {0: '#2ecc71', 1: '#e74c3c'}

print('Libraries loaded.')

In [ ]:
# Load datasets
labels_df = pd.read_csv('../data/processed/labels.csv')
features_df = pd.read_csv('../data/processed/audio_features.csv')

print(f'Labels:  {labels_df.shape[0]} rows, {labels_df.shape[1]} cols')
print(f'Features: {features_df.shape[0]} rows, {features_df.shape[1]} cols')
print()
print('--- Labels ---')
print(labels_df.dtypes)
print()
labels_df.head()

In [ ]:
print('--- Audio Features ---')
print(features_df.dtypes)
print()
features_df.head()

In [ ]:
# Merge labels and features on a common key
# Try multi-column merge on (ticker, date) first, then single-key fallback
if 'ticker' in labels_df.columns and 'ticker' in features_df.columns and \
   'date' in labels_df.columns and 'date' in features_df.columns:
    df = labels_df.merge(features_df, on=['ticker', 'date'], how='inner',
                         suffixes=('', '_feat'))
    merge_key = 'ticker+date'
    print(f'Merged on (ticker, date): {len(df)} rows')
else:
    merge_key = None
    for candidate in ['call_id', 'id', 'filename', 'file_id', 'ticker_date']:
        if candidate in labels_df.columns and candidate in features_df.columns:
            merge_key = candidate
            break
    if merge_key:
        df = labels_df.merge(features_df, on=merge_key, how='inner')
        print(f'Merged on "{merge_key}": {len(df)} rows')
    else:
        df = pd.concat([labels_df.reset_index(drop=True),
                        features_df.reset_index(drop=True)], axis=1)
        print(f'No common key found — concatenated side-by-side: {len(df)} rows')

# Identify the label column
label_col = None
for candidate in ['label', 'missed', 'target', 'y']:
    if candidate in df.columns:
        label_col = candidate
        break
if label_col is None:
    raise ValueError(f'Could not find label column. Columns: {list(df.columns)}')

# Identify numeric feature columns (exclude metadata & label)
meta_cols = {'call_id', 'id', 'filename', 'file_id', 'ticker_date',
             'ticker', 'company', 'date', 'quarter', 'year',
             label_col}
feature_cols = [c for c in df.columns
                if c not in meta_cols and pd.api.types.is_numeric_dtype(df[c])]

print(f'Label column: "{label_col}"')
print(f'Feature columns ({len(feature_cols)}): {feature_cols}')
print(f'Missing values:\n{df[feature_cols + [label_col]].isnull().sum()}')

## 2. Label Distribution

In [ ]:
# Overall label counts
label_counts = df[label_col].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(
    [LABEL_NAMES.get(i, str(i)) for i in label_counts.index],
    label_counts.values,
    color=[COLORS.get(i, '#999') for i in label_counts.index],
    edgecolor='white', linewidth=1.5
)
for bar, count in zip(bars, label_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(count), ha='center', va='bottom', fontweight='bold')

ax.set_title('Label Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

# Class balance
majority = label_counts.max()
minority = label_counts.min()
print(f'Class balance ratio (majority / minority): {majority / minority:.2f}')
print(f'Minority class proportion: {minority / len(df):.1%}')

In [ ]:
# Label distribution by ticker / company (top 20)
ticker_col = None
for candidate in ['ticker', 'company', 'symbol']:
    if candidate in df.columns:
        ticker_col = candidate
        break

if ticker_col:
    top_tickers = df[ticker_col].value_counts().head(20).index
    subset = df[df[ticker_col].isin(top_tickers)]

    fig, ax = plt.subplots(figsize=(14, 6))
    ct = pd.crosstab(subset[ticker_col], subset[label_col])
    ct = ct.reindex(columns=[0, 1], fill_value=0)
    ct.columns = [LABEL_NAMES.get(c, str(c)) for c in ct.columns]
    ct.sort_values(ct.columns[0], ascending=False).plot.bar(
        stacked=True, color=[COLORS[0], COLORS[1]], ax=ax,
        edgecolor='white', linewidth=0.5
    )
    ax.set_title(f'Label Distribution by {ticker_col.title()} (Top 20)')
    ax.set_ylabel('Count')
    ax.legend(title='Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No ticker/company column found — skipping per-company breakdown.')

## 3. Audio Feature Distributions

In [ ]:
# Overlapping histograms for each feature, colored by label
n_features = len(feature_cols)
n_cols = 4
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    for label_val in sorted(df[label_col].unique()):
        subset = df[df[label_col] == label_val][col].dropna()
        ax.hist(subset, bins=30, alpha=0.5, label=LABEL_NAMES.get(label_val, str(label_val)),
                color=COLORS.get(label_val, '#999'), density=True)
    ax.set_title(col, fontsize=11)
    ax.legend(fontsize=8)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Audio Feature Distributions by Label', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots of features grouped by label
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    data_to_plot = [df[df[label_col] == lbl][col].dropna() for lbl in sorted(df[label_col].unique())]
    bp = ax.boxplot(data_to_plot, patch_artist=True,
                    labels=[LABEL_NAMES.get(lbl, str(lbl)) for lbl in sorted(df[label_col].unique())])
    for patch, lbl in zip(bp['boxes'], sorted(df[label_col].unique())):
        patch.set_facecolor(COLORS.get(lbl, '#999'))
        patch.set_alpha(0.6)
    ax.set_title(col, fontsize=11)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Box Plots by Label', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Statistical tests: Mann-Whitney U (non-parametric) for each feature
test_results = []

for col in feature_cols:
    group0 = df[df[label_col] == 0][col].dropna()
    group1 = df[df[label_col] == 1][col].dropna()

    if len(group0) < 5 or len(group1) < 5:
        continue

    # Mann-Whitney U
    u_stat, u_pval = stats.mannwhitneyu(group0, group1, alternative='two-sided')
    # Also run t-test for reference
    t_stat, t_pval = stats.ttest_ind(group0, group1, equal_var=False)
    # Effect size (Cohen's d)
    pooled_std = np.sqrt((group0.std()**2 + group1.std()**2) / 2)
    cohens_d = (group0.mean() - group1.mean()) / pooled_std if pooled_std > 0 else 0

    test_results.append({
        'feature': col,
        'mean_met': group0.mean(),
        'mean_missed': group1.mean(),
        'cohens_d': cohens_d,
        'mann_whitney_U': u_stat,
        'mw_pvalue': u_pval,
        't_stat': t_stat,
        't_pvalue': t_pval,
    })

test_df = pd.DataFrame(test_results).sort_values('mw_pvalue')
test_df['significant'] = test_df['mw_pvalue'] < 0.05

print(f"Features with p < 0.05 (Mann-Whitney): "
      f"{test_df['significant'].sum()} / {len(test_df)}")
print()
test_df.style.format({
    'mean_met': '{:.4f}', 'mean_missed': '{:.4f}',
    'cohens_d': '{:.3f}',
    'mann_whitney_U': '{:.0f}', 'mw_pvalue': '{:.4e}',
    't_stat': '{:.3f}', 't_pvalue': '{:.4e}'
}).apply(lambda row: ['background-color: #d4edda' if row['significant'] else '' 
                       for _ in row], axis=1)

## 4. Feature Correlations

In [ ]:
# Correlation heatmap
corr_matrix = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(max(10, len(feature_cols) * 0.7),
                                max(8, len(feature_cols) * 0.6)))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 8})
ax.set_title('Audio Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Top correlated pairs (potential multicollinearity)
pairs = []
for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        r = corr_matrix.iloc[i, j]
        pairs.append((feature_cols[i], feature_cols[j], r))

pairs_df = pd.DataFrame(pairs, columns=['feature_1', 'feature_2', 'correlation'])
pairs_df['abs_corr'] = pairs_df['correlation'].abs()
pairs_df = pairs_df.sort_values('abs_corr', ascending=False)

print('Top 10 correlated feature pairs (watch for multicollinearity):')
print()
print(pairs_df.head(10).to_string(index=False))

high_corr = pairs_df[pairs_df['abs_corr'] > 0.8]
if len(high_corr) > 0:
    print(f'\n⚠️  {len(high_corr)} pairs with |r| > 0.8 — consider dropping one from each pair.')
else:
    print('\n✓ No pairs with |r| > 0.8.')

In [ ]:
# Point-biserial correlation of each feature with the label
pb_results = []
for col in feature_cols:
    valid = df[[col, label_col]].dropna()
    if len(valid) < 5:
        continue
    r, p = pointbiserialr(valid[label_col], valid[col])
    pb_results.append({'feature': col, 'point_biserial_r': r, 'p_value': p})

pb_df = pd.DataFrame(pb_results).sort_values('p_value')

fig, ax = plt.subplots(figsize=(10, max(5, len(feature_cols) * 0.35)))
colors = ['#e74c3c' if p < 0.05 else '#95a5a6' for p in pb_df['p_value']]
ax.barh(pb_df['feature'], pb_df['point_biserial_r'], color=colors, edgecolor='white')
ax.set_xlabel('Point-Biserial Correlation with Label')
ax.set_title('Feature–Label Correlation (red = p < 0.05)')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print(pb_df.to_string(index=False))

## 5. Temporal Analysis

In [ ]:
# Detect date/time column
date_col = None
for candidate in ['date', 'call_date', 'filing_date', 'report_date']:
    if candidate in df.columns:
        date_col = candidate
        break

if date_col:
    df['_date'] = pd.to_datetime(df[date_col], errors='coerce')
    df['_quarter'] = df['_date'].dt.to_period('Q')
    has_time = True
elif 'year' in df.columns and 'quarter' in df.columns:
    df['_quarter'] = df.apply(lambda r: pd.Period(year=int(r['year']),
                                                   quarter=int(r['quarter']),
                                                   freq='Q'), axis=1)
    has_time = True
else:
    has_time = False
    print('No date or year/quarter columns found — skipping temporal analysis.')

if has_time:
    print(f'Temporal range: {df["_quarter"].min()} → {df["_quarter"].max()}')
    print(f'Quarters represented: {df["_quarter"].nunique()}')

In [ ]:
# Label distribution over time
if has_time:
    quarterly_labels = df.groupby('_quarter')[label_col].value_counts().unstack(fill_value=0)
    quarterly_labels.columns = [LABEL_NAMES.get(c, str(c)) for c in quarterly_labels.columns]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Stacked bar
    quarterly_labels.plot.bar(stacked=True, color=[COLORS[0], COLORS[1]],
                              ax=axes[0], edgecolor='white', linewidth=0.5)
    axes[0].set_title('Label Counts by Quarter')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=45)

    # Miss rate over time
    miss_rate = df.groupby('_quarter')[label_col].mean()
    axes[1].plot(miss_rate.index.astype(str), miss_rate.values,
                marker='o', color=COLORS[1], linewidth=2)
    axes[1].set_title('Miss Rate Over Time')
    axes[1].set_ylabel('Fraction Missed')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].axhline(miss_rate.mean(), ls='--', color='gray', alpha=0.7,
                     label=f'Overall mean: {miss_rate.mean():.2%}')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# Feature trends over time (top 6 most discriminative features)
if has_time and len(test_df) > 0:
    top_features = test_df.head(6)['feature'].tolist()

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for i, feat in enumerate(top_features):
        ax = axes[i]
        for lbl in sorted(df[label_col].unique()):
            sub = df[df[label_col] == lbl].groupby('_quarter')[feat].mean()
            ax.plot(sub.index.astype(str), sub.values,
                    marker='o', label=LABEL_NAMES.get(lbl, str(lbl)),
                    color=COLORS.get(lbl, '#999'), linewidth=1.5, markersize=4)
        ax.set_title(feat, fontsize=11)
        ax.tick_params(axis='x', rotation=45, labelsize=8)
        ax.legend(fontsize=8)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle('Top Discriminative Features Over Time (Quarterly Mean)', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

## 6. Key Findings Summary

### Feature Discrimination
- Features flagged as statistically significant (p < 0.05) in the Mann-Whitney U test are the strongest candidates for the classifier.
- Check the Cohen's d column: |d| > 0.2 is a small effect, > 0.5 medium, > 0.8 large. Features with small effects may still contribute in an ensemble.

### Class Balance
- Review the label distribution above. If the ratio exceeds 3:1, consider:
  - SMOTE or random oversampling for the minority class
  - Class-weighted loss in PyTorch (`weight` arg in `CrossEntropyLoss`)
  - Stratified train/test splits

### Multicollinearity
- Feature pairs with |r| > 0.8 carry redundant information. Consider PCA or dropping one from each pair before training.
- MFCCs are often intercorrelated by design — this is expected.

### Data Quality Notes
- Check for NaN counts above — features with >10% missing may need imputation or exclusion.
- Outliers visible in box plots may warrant winsorization.
- If temporal trends are strong, consider adding quarter/year as a control feature to avoid the model learning market-wide patterns.

### Recommended Next Steps
1. Select the top features (significant + meaningful effect size) for the initial model.
2. Address class imbalance before training.
3. Build the PyTorch Dataset with the merged DataFrame from this notebook.
4. Start with a simple baseline (logistic regression on audio features) before moving to the full multimodal architecture.